In [1]:
import sys
import pandas as pd
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.inventory.policy import (
    calculate_target_inventory,
    calculate_recommended_order_qty,
)

In [2]:
from src.inventory.simulation import evaluate_policy
from src.inventory.simulation import (create_purchase_order, calculate_inventory_position)
from src.inventory.simulation import evaluate_reorder_decision
from src.inventory.simulation import create_policy_order
from src.inventory.simulation import simulate_day_with_policy
from src.inventory.simulation import simulate_inventory_with_policy

In [3]:
total_forecast = 300
safety_stock = 50

target_inventory = calculate_target_inventory(
    total_forecast,
    safety_stock,
)

print("Target inventory:", target_inventory)

Target inventory: 350


In [4]:
inventory_position = 200
reorder_required = True

order_qty = calculate_recommended_order_qty(
    target_inventory=target_inventory,
    inventory_position=inventory_position,
    reorder_required=reorder_required,
)

print("Recommended order quantity:", order_qty)

Recommended order quantity: 150.0


In [5]:
order_qty = calculate_recommended_order_qty(
    target_inventory=target_inventory,
    inventory_position=400,
    reorder_required=False,
)

print("Recommended order quantity:", order_qty)

Recommended order quantity: 0.0


In [6]:
orders = [
    create_purchase_order(
        order_date="2025-01-05",
        quantity=100,
        lead_time_days=2,
    ),
    create_purchase_order(
        order_date="2025-01-08",
        quantity=50,
        lead_time_days=12,
    ),
]

In [7]:
inventory_position = calculate_inventory_position(
    current_stock=80,
    purchase_orders=orders,
    current_date="2025-01-10",
)

print(inventory_position)

130


In [8]:
reorder_required = evaluate_reorder_decision(
    inventory_position=110,
    lead_time_demand=100,
    safety_stock=30,
)

print(reorder_required)

True


In [9]:
reorder_required = evaluate_reorder_decision(
    inventory_position=150,
    lead_time_demand=100,
    safety_stock=30,
)

print(reorder_required)

False


In [10]:
order = create_policy_order(
    date="2025-01-10",
    total_forecast=300,
    lead_time_demand=100,
    safety_stock=30,
    inventory_position=110,
    lead_time_days=4,
)

print(order)

PurchaseOrder(order_date=Timestamp('2025-01-10 00:00:00'), arrival_date=Timestamp('2025-01-14 00:00:00'), quantity=220)


In [11]:
purchase_orders = []

daily_result, new_order = simulate_day_with_policy(
    date="2025-01-10",
    opening_stock=100,
    demand=30,
    total_forecast=300,
    lead_time_demand=100,
    safety_stock=30,
    purchase_orders=purchase_orders,
    lead_time_days=4,
)

print(daily_result)
print(new_order)

{'date': Timestamp('2025-01-10 00:00:00'), 'opening_stock': 100, 'arrival_qty': 0, 'demand': 30, 'units_fulfilled': 30, 'stockout_units': 0, 'closing_stock': 70, 'inventory_position': 70, 'order_qty': 260, 'order_date': Timestamp('2025-01-10 00:00:00'), 'arrival_date': Timestamp('2025-01-14 00:00:00')}
PurchaseOrder(order_date=Timestamp('2025-01-10 00:00:00'), arrival_date=Timestamp('2025-01-14 00:00:00'), quantity=260)


In [12]:
test_demand = pd.DataFrame(
    {
        "date": pd.date_range("2025-01-01", periods=5),
        "units_sold": [30, 40, 50, 20, 10],
    }
)

simulation = simulate_inventory_with_policy(
    demand_df=test_demand,
    initial_stock=100,
    total_forecast=300,
    lead_time_demand=100,
    safety_stock=30,
    lead_time_days=2,
)

simulation

,date,opening_stock,arrival_qty,demand,units_fulfilled,stockout_units,closing_stock,inventory_position,order_qty,order_date,arrival_date
0,2025-01-01,100,0,30,30,0,70,70,260,2025-01-01,2025-01-03
1,2025-01-02,70,0,40,40,0,30,290,0,NaT,NaT
2,2025-01-03,30,260,50,50,0,240,240,0,NaT,NaT
3,2025-01-04,240,0,20,20,0,220,220,0,NaT,NaT
4,2025-01-05,220,0,10,10,0,210,210,0,NaT,NaT


In [13]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

sales = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "sales.csv"
)

sales.head()

,date,product_id,product_name,category,price,discount,promotion,day_of_week,month,is_weekend,units_sold
0,2024-01-01,P001,Wireless Headphones,Electronics,1999.0,0,0,0,1,False,33
1,2024-01-02,P001,Wireless Headphones,Electronics,1999.0,0,0,1,1,False,39
2,2024-01-03,P001,Wireless Headphones,Electronics,1999.0,0,0,2,1,False,39
3,2024-01-04,P001,Wireless Headphones,Electronics,1799.1,10,1,3,1,False,59
4,2024-01-05,P001,Wireless Headphones,Electronics,1999.0,0,0,4,1,False,35


In [14]:
sales["product_id"].unique()

<StringArray>
['P001', 'P002', 'P003', 'P004', 'P005']
Length: 5, dtype: str

In [15]:
product_id = sales["product_id"].iloc[0]

product_sales = (
    sales[sales["product_id"] == product_id]
    .copy()
)

product_sales["date"] = pd.to_datetime(
    product_sales["date"]
)

product_sales = (
    product_sales
    .sort_values("date")
    .reset_index(drop=True)
)

product_sales[
    ["date", "product_id", "units_sold"]
].head(10)

,date,product_id,units_sold
0,2024-01-01,P001,33
1,2024-01-02,P001,39
2,2024-01-03,P001,39
3,2024-01-04,P001,59
4,2024-01-05,P001,35
5,2024-01-06,P001,41
6,2024-01-07,P001,30
7,2024-01-08,P001,42
8,2024-01-09,P001,43
9,2024-01-10,P001,34


In [16]:
test_period = product_sales.head(30)[
    ["date", "units_sold"]
].copy()

test_period.head()

,date,units_sold
0,2024-01-01,33
1,2024-01-02,39
2,2024-01-03,39
3,2024-01-04,59
4,2024-01-05,35


In [17]:
simulation = simulate_inventory_with_policy(
    demand_df=test_period,
    initial_stock=100,
    total_forecast=300,
    lead_time_demand=100,
    safety_stock=30,
    lead_time_days=4,
)

simulation

,date,opening_stock,arrival_qty,demand,units_fulfilled,stockout_units,closing_stock,inventory_position,order_qty,order_date,arrival_date
0,2024-01-01,100,0,33,33,0,67,67,263,2024-01-01,2024-01-05
1,2024-01-02,67,0,39,39,0,28,291,0,NaT,NaT
2,2024-01-03,28,0,39,28,11,0,263,0,NaT,NaT
3,2024-01-04,0,0,59,0,59,0,263,0,NaT,NaT
4,2024-01-05,0,263,35,35,0,228,228,0,NaT,NaT
5,2024-01-06,228,0,41,41,0,187,187,0,NaT,NaT
6,2024-01-07,187,0,30,30,0,157,157,0,NaT,NaT
7,2024-01-08,157,0,42,42,0,115,115,215,2024-01-08,2024-01-12
8,2024-01-09,115,0,43,43,0,72,287,0,NaT,NaT
9,2024-01-10,72,0,34,34,0,38,253,0,NaT,NaT


In [18]:
simulation[
    simulation["order_qty"] > 0
][
    [
        "date",
        "order_qty",
        "order_date",
        "arrival_date",
    ]
]

,date,order_qty,order_date,arrival_date
0,2024-01-01,263,2024-01-01,2024-01-05
7,2024-01-08,215,2024-01-08,2024-01-12
12,2024-01-13,213,2024-01-13,2024-01-17
18,2024-01-19,237,2024-01-19,2024-01-23
23,2024-01-24,238,2024-01-24,2024-01-28
29,2024-01-30,236,2024-01-30,2024-02-03


In [19]:
from src.models.forecasting import forecast_product_demand
from src.inventory.safety_stock import calculate_safety_stock
from src.inventory.policy import calculate_target_inventory
import joblib
from pathlib import Path

In [20]:
MODEL_PATH = PROJECT_ROOT / "models" / "xgboost_forecaster.joblib"

model = joblib.load(MODEL_PATH)

print("Model loaded successfully")
print(type(model))

Model loaded successfully
<class 'xgboost.sklearn.XGBRegressor'>


In [21]:
product_id = "P001"

product_history = (
    sales[sales["product_id"] == product_id]
    .copy()
    .sort_values("date")
)

product_history["date"] = pd.to_datetime(
    product_history["date"]
)

print(product_history[["date", "product_id", "units_sold"]].tail())

          date product_id  units_sold
726 2025-12-27       P001          40
727 2025-12-28       P001          66
728 2025-12-29       P001          45
729 2025-12-30       P001          47
730 2025-12-31       P001          24


In [22]:
MODEL_FEATURES = [
    "price",
    "discount",
    "promotion",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
]

forecast = forecast_product_demand(
    model=model,
    product_history=product_history,
    model_features=MODEL_FEATURES,
    horizon=30,
)

forecast.head()

,date,product_id,forecast_units
0,2026-01-01,P001,36.979706
1,2026-01-02,P001,37.337029
2,2026-01-03,P001,42.004692
3,2026-01-04,P001,41.931583
4,2026-01-05,P001,35.965870


In [23]:
total_forecast = forecast["forecast_units"].sum()

print("Total 30-day forecast:", total_forecast)

Total 30-day forecast: 1044.898847579956


In [24]:
inventory = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "inventory_snapshot.csv"
)

inventory[
    inventory["product_id"] == product_id
]

,snapshot_date,product_id,current_stock,open_order_qty,expected_arrival_date,lead_time_days,unit_cost
0,2025-12-31,P001,225,0,NaN,4,1000


In [25]:
lead_time_days = 4

lead_time_demand = forecast.head(
    lead_time_days
)["forecast_units"].sum()

print("Lead-time demand:", lead_time_demand)

Lead-time demand: 158.25300979614258


In [26]:
safety_stock = 30

target_inventory = calculate_target_inventory(
    total_forecast=total_forecast,
    safety_stock=safety_stock,
)

print("30-day target inventory:", target_inventory)

30-day target inventory: 1074.898847579956


In [27]:
p001_inventory = inventory[
    inventory["product_id"] == product_id
].iloc[0]

print(p001_inventory)

snapshot_date            2025-12-31
product_id                     P001
current_stock                   225
open_order_qty                    0
expected_arrival_date           NaN
lead_time_days                    4
unit_cost                      1000
Name: 0, dtype: object


In [28]:
current_stock = int(p001_inventory["current_stock"])
open_order_qty = int(p001_inventory["open_order_qty"])
lead_time_days = int(p001_inventory["lead_time_days"])
unit_cost = float(p001_inventory["unit_cost"])

print("Current stock:", current_stock)
print("Open order quantity:", open_order_qty)
print("Lead time:", lead_time_days)
print("Unit cost:", unit_cost)

Current stock: 225
Open order quantity: 0
Lead time: 4
Unit cost: 1000.0


In [29]:
from src.inventory.position import calculate_inventory_position

inventory_position = calculate_inventory_position(
    current_stock=current_stock,
    open_order_qty=open_order_qty,
)

print("Inventory position:", inventory_position)

Inventory position: 225


In [30]:
forecast_error_std = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "forecast_error_std.csv"
)

forecast_error_std

,product_id,error_std
0,P001,5.594148
1,P002,3.845135
2,P003,4.801437
3,P004,2.772442
4,P005,6.277869


In [31]:
p001_error_std = float(
    forecast_error_std.loc[
        forecast_error_std["product_id"] == product_id,
        "error_std"
    ].iloc[0]
)

print("P001 forecast error std:", p001_error_std)

P001 forecast error std: 5.594148419718317


In [32]:
safety_stock = calculate_safety_stock(
    error_std=p001_error_std,
    lead_time_days=lead_time_days,
)

print("Safety stock:", safety_stock)

Safety stock: 19.0


In [33]:
from src.inventory.reorder import calculate_reorder_point

reorder_point = calculate_reorder_point(
    lead_time_demand=lead_time_demand,
    safety_stock=safety_stock,
)

print("Reorder point:", reorder_point)

Reorder point: 177.25300979614258


In [34]:
from src.inventory.reorder import should_reorder

reorder_required = should_reorder(
    inventory_position=inventory_position,
    reorder_point=reorder_point,
)

print("Reorder required:", reorder_required)

Reorder required: False


In [35]:

from src.inventory.policy import calculate_recommended_order_qty

order_qty = calculate_recommended_order_qty(
    target_inventory=target_inventory,
    inventory_position=inventory_position,
    reorder_required=reorder_required,
)

order_qty = float(np.asarray(order_qty).item())

print("Recommended order quantity:", order_qty)

Recommended order quantity: 0.0


In [36]:
backtest_date = pd.Timestamp("2025-10-01")

In [37]:
history_before_backtest = (
    product_history[
        product_history["date"] < backtest_date
    ]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

print("Last historical date available:")
print(history_before_backtest["date"].max())

Last historical date available:
2025-09-30 00:00:00


In [38]:
forecast = forecast_product_demand(
    model=model,
    product_history=history_before_backtest,
    model_features=MODEL_FEATURES,
    horizon=30,
)

print(forecast.head())

        date product_id  forecast_units
0 2025-10-01       P001       32.474064
1 2025-10-02       P001       32.357204
2 2025-10-03       P001       32.738224
3 2025-10-04       P001       36.781902
4 2025-10-05       P001       36.063351


In [39]:
total_forecast = forecast["forecast_units"].sum()

lead_time_demand = forecast.head(
    lead_time_days
)["forecast_units"].sum()

target_inventory = calculate_target_inventory(
    total_forecast=total_forecast,
    safety_stock=safety_stock,
)

reorder_point = calculate_reorder_point(
    lead_time_demand=lead_time_demand,
    safety_stock=safety_stock,
)

reorder_required = should_reorder(
    inventory_position=inventory_position,
    reorder_point=reorder_point,
)

order_qty = calculate_recommended_order_qty(
    target_inventory=target_inventory,
    inventory_position=inventory_position,
    reorder_required=reorder_required,
)

order_qty = float(np.asarray(order_qty).item())

In [40]:
print("Total forecast:", total_forecast)
print("Lead-time demand:", lead_time_demand)
print("Safety stock:", safety_stock)
print("Target inventory:", target_inventory)
print("Reorder point:", reorder_point)
print("Inventory position:", inventory_position)
print("Reorder required:", reorder_required)
print("Order quantity:", order_qty)

Total forecast: 831.9366607666016
Lead-time demand: 134.3513946533203
Safety stock: 19.0
Target inventory: 850.9366607666016
Reorder point: 153.3513946533203
Inventory position: 225
Reorder required: False
Order quantity: 0.0


In [41]:
actual_demand = int(
    sales.loc[
        (sales["product_id"] == product_id)
        & (pd.to_datetime(sales["date"]) == backtest_date),
        "units_sold"
    ].iloc[0]
)

print("Actual October 1 demand:", actual_demand)

Actual October 1 demand: 39


In [42]:
from src.inventory.simulation import process_daily_demand

opening_stock = current_stock

units_fulfilled, stockout_units, closing_stock = process_daily_demand(
    available_stock=opening_stock,
    demand=actual_demand,
)

print("Opening stock:", opening_stock)
print("Actual demand:", actual_demand)
print("Units fulfilled:", units_fulfilled)
print("Stockout units:", stockout_units)
print("Closing stock:", closing_stock)

Opening stock: 225
Actual demand: 39
Units fulfilled: 39
Stockout units: 0
Closing stock: 186


In [43]:
backtest_date_2 = pd.Timestamp("2025-10-02")

opening_stock_2 = closing_stock

print("October 2 opening stock:", opening_stock_2)

October 2 opening stock: 186


In [44]:
actual_demand_2 = int(
    sales.loc[
        (sales["product_id"] == product_id)
        & (pd.to_datetime(sales["date"]) == backtest_date_2),
        "units_sold"
    ].iloc[0]
)

print("Actual October 2 demand:", actual_demand_2)

Actual October 2 demand: 27


In [45]:
units_fulfilled_2, stockout_units_2, closing_stock_2 = process_daily_demand(
    available_stock=opening_stock_2,
    demand=actual_demand_2,
)

print("Opening stock:", opening_stock_2)
print("Actual demand:", actual_demand_2)
print("Units fulfilled:", units_fulfilled_2)
print("Stockout units:", stockout_units_2)
print("Closing stock:", closing_stock_2)

Opening stock: 186
Actual demand: 27
Units fulfilled: 27
Stockout units: 0
Closing stock: 159


In [46]:
backtest_start = pd.Timestamp("2025-10-01")

historical_before_backtest = sales[
    (sales["product_id"] == product_id)
    & (pd.to_datetime(sales["date"]) < backtest_start)
].copy()

average_daily_demand = (
    historical_before_backtest["units_sold"].mean()
)

inventory_days = 5  # V1 configuration for P001

starting_stock = int(
    round(average_daily_demand * inventory_days)
)

print("Average daily demand before backtest:", average_daily_demand)
print("V1 inventory days:", inventory_days)
print("V2 starting stock:", starting_stock)

Average daily demand before backtest: 45.41627543035994
V1 inventory days: 5
V2 starting stock: 227


In [47]:
from src.inventory.simulation import simulate_backtest_day

In [48]:
purchase_orders = []

result_oct1, new_order = simulate_backtest_day(
    current_date=pd.Timestamp("2025-10-01"),
    product_history=sales[
        sales["product_id"] == product_id
    ].copy(),
    current_stock=starting_stock,
    purchase_orders=purchase_orders,
    model=model,
    model_features=MODEL_FEATURES,
    safety_stock=safety_stock,
    lead_time_days=lead_time_days,
)

print(result_oct1)
print("New order:", new_order)

{'date': Timestamp('2025-10-01 00:00:00'), 'opening_stock': 227, 'arrival_qty': 0, 'demand': 39, 'units_fulfilled': 39, 'stockout_units': 0, 'closing_stock': 188, 'total_forecast': 831.9366607666016, 'lead_time_demand': 134.3513946533203, 'safety_stock': np.float64(19.0), 'reorder_point': np.float64(153.3513946533203), 'inventory_position': 188, 'reorder_required': np.False_, 'target_inventory': np.float64(850.9366607666016), 'order_qty': 0}
New order: None


In [49]:
purchase_orders = []

# Carry forward any order created on October 1.
if new_order is not None:
    purchase_orders.append(new_order)

result_oct2, new_order_oct2 = simulate_backtest_day(
    current_date=pd.Timestamp("2025-10-02"),
    product_history=sales[
        sales["product_id"] == product_id
    ].copy(),
    current_stock=result_oct1["closing_stock"],
    purchase_orders=purchase_orders,
    model=model,
    model_features=MODEL_FEATURES,
    safety_stock=safety_stock,
    lead_time_days=lead_time_days,
)

print(result_oct2)
print("New order:", new_order_oct2)

{'date': Timestamp('2025-10-02 00:00:00'), 'opening_stock': 188, 'arrival_qty': 0, 'demand': 27, 'units_fulfilled': 27, 'stockout_units': 0, 'closing_stock': 161, 'total_forecast': 835.2193946838379, 'lead_time_demand': 140.27993774414062, 'safety_stock': np.float64(19.0), 'reorder_point': np.float64(159.27993774414062), 'inventory_position': 161, 'reorder_required': np.False_, 'target_inventory': np.float64(854.2193946838379), 'order_qty': 0}
New order: None


In [50]:
from src.inventory.simulation import run_backtest

In [51]:
backtest_results = run_backtest(
    product_history=sales[
        sales["product_id"] == product_id
    ].copy(),
    start_date="2025-10-01",
    end_date="2025-12-31",
    starting_stock=starting_stock,
    model=model,
    model_features=MODEL_FEATURES,
    safety_stock=safety_stock,
    lead_time_days=lead_time_days,
)

In [52]:
backtest_results.head(10)

,date,opening_stock,arrival_qty,demand,units_fulfilled,stockout_units,closing_stock,total_forecast,lead_time_demand,safety_stock,reorder_point,inventory_position,reorder_required,target_inventory,order_qty
0,2025-10-01,227,0,39,39,0,188,831.936661,134.351395,19.0,153.351395,188,False,850.936661,0
1,2025-10-02,188,0,27,27,0,161,835.219395,140.279938,19.0,159.279938,161,False,854.219395,0
2,2025-10-03,161,0,45,45,0,116,821.565731,138.809292,19.0,157.809292,116,True,840.565731,725
3,2025-10-04,116,0,40,40,0,76,833.714184,137.855717,19.0,156.855717,801,False,852.714184,0
4,2025-10-05,76,0,35,35,0,41,820.384241,131.314219,19.0,150.314219,766,False,839.384241,0
5,2025-10-06,41,0,36,36,0,5,793.311888,123.703941,19.0,142.703941,730,False,812.311888,0
6,2025-10-07,5,725,29,29,0,701,793.535791,123.470861,19.0,142.470861,701,False,812.535791,0
7,2025-10-08,701,0,40,40,0,661,788.257832,125.822227,19.0,144.822227,661,False,807.257832,0
8,2025-10-09,661,0,46,46,0,615,802.409151,130.099754,19.0,149.099754,615,False,821.409151,0
9,2025-10-10,615,0,53,53,0,562,819.332996,130.072737,19.0,149.072737,562,False,838.332996,0


In [53]:
backtest_results.tail(10)

,date,opening_stock,arrival_qty,demand,units_fulfilled,stockout_units,closing_stock,total_forecast,lead_time_demand,safety_stock,reorder_point,inventory_position,reorder_required,target_inventory,order_qty
82,2025-12-22,784,0,64,64,0,720,1009.781830,140.467720,19.0,159.467720,720,False,1028.781830,0
83,2025-12-23,720,0,60,60,0,660,1047.455584,143.710789,19.0,162.710789,660,False,1066.455584,0
84,2025-12-24,660,0,38,38,0,622,1070.465521,148.829586,19.0,167.829586,622,False,1089.465521,0
85,2025-12-25,622,0,30,30,0,592,1051.677742,152.541580,19.0,171.541580,592,False,1070.677742,0
86,2025-12-26,592,0,47,47,0,545,1040.954161,149.669945,19.0,168.669945,545,False,1059.954161,0
87,2025-12-27,545,0,40,40,0,505,1070.363373,152.958271,19.0,171.958271,505,False,1089.363373,0
88,2025-12-28,505,0,66,66,0,439,1041.602358,146.082504,19.0,165.082504,439,False,1060.602358,0
89,2025-12-29,439,0,45,45,0,394,1078.834309,146.752975,19.0,165.752975,394,False,1097.834309,0
90,2025-12-30,394,0,47,47,0,347,1063.086887,147.303947,19.0,166.303947,347,False,1082.086887,0
91,2025-12-31,347,0,24,24,0,323,1080.398159,155.113087,19.0,174.113087,323,False,1099.398159,0


In [54]:

backtest_results[backtest_results["stockout_units"] > 0]

,date,opening_stock,arrival_qty,demand,units_fulfilled,stockout_units,closing_stock,total_forecast,lead_time_demand,safety_stock,reorder_point,inventory_position,reorder_required,target_inventory,order_qty
24,2025-10-25,27,0,42,27,15,0,855.039806,125.439156,19.0,144.439156,696,False,874.039806,0
79,2025-12-19,47,0,58,47,11,0,1012.212479,149.377411,19.0,168.377411,886,False,1031.212479,0


In [55]:
backtest_results.shape

(92, 15)

In [56]:
backtest_results[
    backtest_results["order_qty"] > 0
].head(10)


,date,opening_stock,arrival_qty,demand,units_fulfilled,stockout_units,closing_stock,total_forecast,lead_time_demand,safety_stock,reorder_point,inventory_position,reorder_required,target_inventory,order_qty
2,2025-10-03,161,0,45,45,0,116,821.565731,138.809292,19.0,157.809292,116,True,840.565731,725
21,2025-10-22,161,0,43,43,0,118,794.846090,115.838181,19.0,134.838181,118,True,813.846090,696
38,2025-11-08,167,0,39,39,0,128,939.609661,138.534842,19.0,157.534842,128,True,958.609661,831
57,2025-11-27,221,0,63,63,0,158,949.395458,146.371693,19.0,165.371693,158,True,968.395458,811
76,2025-12-16,165,0,38,38,0,127,993.039553,138.061806,19.0,157.061806,127,True,1012.039553,886


In [57]:
backtest_results[
    (backtest_results["date"] >= "2025-10-03")
    & (backtest_results["date"] <= "2025-10-09")
][
    [
        "date",
        "opening_stock",
        "arrival_qty",
        "demand",
        "closing_stock",
        "order_qty",
    ]
]

,date,opening_stock,arrival_qty,demand,closing_stock,order_qty
2,2025-10-03,161,0,45,116,725
3,2025-10-04,116,0,40,76,0
4,2025-10-05,76,0,35,41,0
5,2025-10-06,41,0,36,5,0
6,2025-10-07,5,725,29,701,0
7,2025-10-08,701,0,40,661,0
8,2025-10-09,661,0,46,615,0


In [58]:
average_inventory = backtest_results["closing_stock"].mean()

maximum_inventory = backtest_results["closing_stock"].max()

total_units_ordered = backtest_results["order_qty"].sum()

number_of_orders = (
    backtest_results["order_qty"] > 0
).sum()

print("Average inventory:", round(average_inventory, 2))
print("Maximum inventory:", maximum_inventory)
print("Total units ordered:", total_units_ordered)
print("Number of orders:", number_of_orders)

Average inventory: 374.92
Maximum inventory: 822
Total units ordered: 3949
Number of orders: 5


In [59]:
stockout_days = (
    backtest_results["stockout_units"] > 0
).sum()

total_stockout_units = (
    backtest_results["stockout_units"].sum()
)

print("Stockout days:", stockout_days)
print("Total lost sales / stockout units:", total_stockout_units)

Stockout days: 2
Total lost sales / stockout units: 26


In [60]:
total_demand = backtest_results["demand"].sum()

total_fulfilled = (
    backtest_results["units_fulfilled"].sum()
)

service_level = (
    total_fulfilled / total_demand
    if total_demand > 0
    else 0
)

print("Total demand:", total_demand)
print("Total fulfilled:", total_fulfilled)
print("Service level:", round(service_level * 100, 2), "%")

Total demand: 3879
Total fulfilled: 3853
Service level: 99.33 %


In [61]:
holding_cost_rate = 0.20  # 20% annual holding cost

annual_inventory_value = (
    backtest_results["closing_stock"]
    * unit_cost
).mean()

holding_cost = (
    annual_inventory_value
    * holding_cost_rate
    * len(backtest_results)
    / 365
)

print("Average inventory value:", round(annual_inventory_value, 2))
print("92-day holding cost:", round(holding_cost, 2))

Average inventory value: 374923.91
92-day holding cost: 18900.27


In [62]:
ordering_cost_per_order = 500

ordering_cost = (
    number_of_orders
    * ordering_cost_per_order
)

print("Number of orders:", number_of_orders)
print("Ordering cost:", round(ordering_cost, 2))

Number of orders: 5
Ordering cost: 2500


In [63]:
stockout_cost_per_unit = 1000

stockout_cost = (
    total_stockout_units
    * stockout_cost_per_unit
)

print("Stockout units:", total_stockout_units)
print("Stockout cost:", round(stockout_cost, 2))

Stockout units: 26
Stockout cost: 26000


In [64]:
total_inventory_cost = (
    holding_cost
    + ordering_cost
    + stockout_cost
)

print("Holding cost:", round(holding_cost, 2))
print("Ordering cost:", round(ordering_cost, 2))
print("Stockout cost:", round(stockout_cost, 2))
print("Total inventory cost:", round(total_inventory_cost, 2))

Holding cost: 18900.27
Ordering cost: 2500
Stockout cost: 26000
Total inventory cost: 47400.27


In [65]:
baseline_history = (
    sales[
        (sales["product_id"] == product_id)
        & (pd.to_datetime(sales["date"]) < backtest_date)
    ]
    .copy()
    .sort_values("date")
)

baseline_history["date"] = pd.to_datetime(
    baseline_history["date"]
)

baseline_forecast = (
    baseline_history["units_sold"]
    .tail(7)
    .mean()
)

print("P001 baseline forecast for", backtest_date.date(), ":",
      round(baseline_forecast, 2))

P001 baseline forecast for 2025-10-01 : 33.14


In [66]:
print("Backtest date:", backtest_date.date())
print("Product:", product_id)

print(
    "XGBoost 30-day forecast:",
    round(total_forecast, 2)
)

print(
    "XGBoost daily average forecast:",
    round(
        forecast["forecast_units"].mean(),
        2
    )
)

print(
    "Moving Average daily forecast:",
    round(
        baseline_forecast,
        2
    )
)

Backtest date: 2025-10-01
Product: P001
XGBoost 30-day forecast: 831.94
XGBoost daily average forecast: 27.73
Moving Average daily forecast: 33.14


In [67]:
baseline_total_forecast = (
    baseline_forecast * 30
)

print(
    "Baseline daily forecast:",
    round(baseline_forecast, 2)
)

print(
    "Baseline 30-day forecast:",
    round(baseline_total_forecast, 2)
)

Baseline daily forecast: 33.14
Baseline 30-day forecast: 994.29


In [68]:
baseline_lead_time_demand = (
    baseline_forecast * lead_time_days
)

print(
    "Baseline lead-time demand:",
    round(baseline_lead_time_demand, 2)
)

Baseline lead-time demand: 132.57


In [69]:
baseline_reorder_point = calculate_reorder_point(
    lead_time_demand=baseline_lead_time_demand,
    safety_stock=safety_stock,
)

print(
    "Baseline reorder point:",
    round(baseline_reorder_point, 2)
)

Baseline reorder point: 151.57


In [70]:
baseline_target_inventory = calculate_target_inventory(
    total_forecast=baseline_total_forecast,
    safety_stock=safety_stock,
)

baseline_reorder_required = should_reorder(
    inventory_position=inventory_position,
    reorder_point=baseline_reorder_point,
)

baseline_order_qty = calculate_recommended_order_qty(
    target_inventory=baseline_target_inventory,
    inventory_position=inventory_position,
    reorder_required=baseline_reorder_required,
)

baseline_order_qty = float(
    np.asarray(baseline_order_qty).item()
)

print("Baseline target inventory:", round(baseline_target_inventory, 2))
print("Baseline reorder required:", baseline_reorder_required)
print("Baseline order quantity:", baseline_order_qty)

Baseline target inventory: 1013.29
Baseline reorder required: False
Baseline order quantity: 0.0


In [71]:
print(product_history["date"].min())
print(product_history["date"].max())
print(product_history["date"].dtype)

2024-01-01 00:00:00
2025-12-31 00:00:00
datetime64[us]


In [72]:
print("Sales date range:")
print(sales["date"].min())
print(sales["date"].max())

print("\nProduct history date range:")
print(product_history["date"].min())
print(product_history["date"].max())

print("\nSales shape:", sales.shape)
print("Product history shape:", product_history.shape)

Sales date range:
2024-01-01
2025-12-31

Product history date range:
2024-01-01 00:00:00
2025-12-31 00:00:00

Sales shape: (3655, 11)
Product history shape: (731, 11)


In [73]:
print(product_history["date"].min())
print(product_history["date"].max())
print(history_before_backtest["date"].min())
print(history_before_backtest["date"].max())

2024-01-01 00:00:00
2025-12-31 00:00:00
2024-01-01 00:00:00
2025-09-30 00:00:00


In [74]:
backtest_start = pd.Timestamp("2025-10-01")
backtest_end = pd.Timestamp("2025-12-31")

backtest_days = product_history[
    (product_history["date"] >= backtest_start)
    & (product_history["date"] <= backtest_end)
].copy()

print("Backtest rows:", len(backtest_days))
print("Backtest start:", backtest_days["date"].min())
print("Backtest end:", backtest_days["date"].max())

Backtest rows: 92
Backtest start: 2025-10-01 00:00:00
Backtest end: 2025-12-31 00:00:00


In [75]:
from src.inventory.simulation import (
    create_purchase_order,
    calculate_inventory_position,
    get_arrivals_for_date,
    process_daily_demand,
)



baseline_results = []

current_stock_baseline = starting_stock
purchase_orders_baseline = []

for current_date in backtest_days["date"]:

    # Use only information available before today
    history_before_today = product_history[
        product_history["date"] < current_date
    ].copy()

    # 7-day moving average forecast
    baseline_daily_forecast = (
        history_before_today["units_sold"]
        .tail(7)
        .mean()
    )

    baseline_total_forecast = baseline_daily_forecast * 30

    baseline_lead_time_demand = (
        baseline_daily_forecast * lead_time_days
    )

    # Receive orders arriving today
    arrival_qty = get_arrivals_for_date(
        purchase_orders=purchase_orders_baseline,
        date=current_date,
    )

    available_stock = (
        current_stock_baseline + arrival_qty
    )

    # Actual historical demand for today
    actual_demand = int(
        product_history.loc[
            product_history["date"] == current_date,
            "units_sold"
        ].iloc[0]
    )

    # Consume demand
    (
        units_fulfilled,
        stockout_units,
        closing_stock,
    ) = process_daily_demand(
        available_stock=available_stock,
        demand=actual_demand,
    )

    # Inventory position after today's demand
    inventory_position_baseline = calculate_inventory_position(
        current_stock=closing_stock,
        purchase_orders=purchase_orders_baseline,
        current_date=current_date,
    )

    # Reorder decision
    baseline_reorder_point = calculate_reorder_point(
        lead_time_demand=baseline_lead_time_demand,
        safety_stock=safety_stock,
    )

    baseline_reorder_required = should_reorder(
        inventory_position=inventory_position_baseline,
        reorder_point=baseline_reorder_point,
    )

    # Target inventory
    baseline_target_inventory = calculate_target_inventory(
        total_forecast=baseline_total_forecast,
        safety_stock=safety_stock,
    )

    # Order quantity
    baseline_order_qty = calculate_recommended_order_qty(
        target_inventory=baseline_target_inventory,
        inventory_position=inventory_position_baseline,
        reorder_required=baseline_reorder_required,
    )

    baseline_order_qty = float(
        np.asarray(baseline_order_qty).item()
    )

    # Create purchase order if required
    new_order = None

    if baseline_order_qty > 0:
        new_order = create_purchase_order(
            order_date=current_date,
            quantity=int(baseline_order_qty),
            lead_time_days=lead_time_days,
        )

        purchase_orders_baseline.append(new_order)

    # Store daily result
    baseline_results.append(
        {
            "date": current_date,
            "opening_stock": current_stock_baseline,
            "arrival_qty": arrival_qty,
            "demand": actual_demand,
            "units_fulfilled": units_fulfilled,
            "stockout_units": stockout_units,
            "closing_stock": closing_stock,
            "baseline_daily_forecast": baseline_daily_forecast,
            "baseline_total_forecast": baseline_total_forecast,
            "baseline_lead_time_demand": baseline_lead_time_demand,
            "safety_stock": safety_stock,
            "baseline_reorder_point": baseline_reorder_point,
            "inventory_position": inventory_position_baseline,
            "baseline_reorder_required": baseline_reorder_required,
            "baseline_target_inventory": baseline_target_inventory,
            "order_qty": int(baseline_order_qty),
        }
    )

    # Tomorrow starts with today's closing stock
    current_stock_baseline = closing_stock

baseline_results = pd.DataFrame(baseline_results)

print("Baseline backtest shape:", baseline_results.shape)
baseline_results.head()

Baseline backtest shape: (92, 16)


,date,opening_stock,arrival_qty,demand,units_fulfilled,stockout_units,closing_stock,baseline_daily_forecast,baseline_total_forecast,baseline_lead_time_demand,safety_stock,baseline_reorder_point,inventory_position,baseline_reorder_required,baseline_target_inventory,order_qty
0,2025-10-01,227,0,39,39,0,188,33.142857,994.285714,132.571429,19.0,151.571429,188,False,1013.285714,0
1,2025-10-02,188,0,27,27,0,161,31.857143,955.714286,127.428571,19.0,146.428571,161,False,974.714286,0
2,2025-10-03,161,0,45,45,0,116,31.571429,947.142857,126.285714,19.0,145.285714,116,True,966.142857,851
3,2025-10-04,116,0,40,40,0,76,33.714286,1011.428571,134.857143,19.0,153.857143,927,False,1030.428571,0
4,2025-10-05,76,0,35,35,0,41,34.000000,1020.000000,136.000000,19.0,155.000000,892,False,1039.000000,0


In [76]:
print("Baseline backtest shape:", baseline_results.shape)

Baseline backtest shape: (92, 16)


In [77]:
# Baseline performance metrics

baseline_avg_inventory = baseline_results["closing_stock"].mean()
baseline_max_inventory = baseline_results["closing_stock"].max()

baseline_total_units_ordered = baseline_results["order_qty"].sum()
baseline_number_of_orders = (
    baseline_results["order_qty"] > 0
).sum()

baseline_stockout_days = (
    baseline_results["stockout_units"] > 0
).sum()

baseline_total_lost_sales = (
    baseline_results["stockout_units"].sum()
)

baseline_total_demand = (
    baseline_results["demand"].sum()
)

baseline_total_fulfilled = (
    baseline_results["units_fulfilled"].sum()
)

baseline_service_level = (
    baseline_total_fulfilled
    / baseline_total_demand
    * 100
)

print("Baseline Performance")
print("--------------------")
print("Average inventory:", round(baseline_avg_inventory, 2))
print("Maximum inventory:", baseline_max_inventory)
print("Total units ordered:", baseline_total_units_ordered)
print("Number of orders:", baseline_number_of_orders)
print("Stockout days:", baseline_stockout_days)
print("Lost sales:", baseline_total_lost_sales)
print("Total demand:", baseline_total_demand)
print("Units fulfilled:", baseline_total_fulfilled)
print("Service level:", round(baseline_service_level, 2), "%")

Baseline Performance
--------------------
Average inventory: 544.41
Maximum inventory: 1261
Total units ordered: 4139
Number of orders: 4
Stockout days: 0
Lost sales: 0
Total demand: 3879
Units fulfilled: 3879
Service level: 100.0 %


In [78]:
# V2 cost assumptions

holding_cost_rate = 0.20          # 20% of inventory value per year
ordering_cost_per_order = 500     # ₹500 per purchase order
stockout_cost_per_unit = 1000     # ₹1,000 per lost unit

unit_cost = 1000                  # P001 unit cost from V1
backtest_days_count = len(backtest_results)

print("Holding cost rate:", holding_cost_rate)
print("Ordering cost per order: ₹", ordering_cost_per_order)
print("Stockout cost per unit: ₹", stockout_cost_per_unit)
print("Unit cost: ₹", unit_cost)
print("Backtest days:", backtest_days_count)

Holding cost rate: 0.2
Ordering cost per order: ₹ 500
Stockout cost per unit: ₹ 1000
Unit cost: ₹ 1000
Backtest days: 92


In [79]:
# XGBoost inventory cost

average_inventory_value = average_inventory * unit_cost

holding_cost = (
    average_inventory_value
    * holding_cost_rate
    * backtest_days_count
    / 365
)

ordering_cost = (
    number_of_orders
    * ordering_cost_per_order
)

total_lost_units = backtest_results["stockout_units"].sum()

stockout_cost = (
    total_lost_units
    * stockout_cost_per_unit
)

total_inventory_cost = (
    holding_cost
    + ordering_cost
    + stockout_cost
)

print("XGBoost Cost")
print("------------")
print("Holding cost: ₹", round(holding_cost, 2))
print("Ordering cost: ₹", round(ordering_cost, 2))
print("Stockout cost: ₹", round(stockout_cost, 2))
print("Total inventory cost: ₹", round(total_inventory_cost, 2))

XGBoost Cost
------------
Holding cost: ₹ 18900.27
Ordering cost: ₹ 2500
Stockout cost: ₹ 26000
Total inventory cost: ₹ 47400.27


In [80]:
# Moving Average inventory cost

baseline_average_inventory_value = (
    baseline_avg_inventory * unit_cost
)

baseline_holding_cost = (
    baseline_average_inventory_value
    * holding_cost_rate
    * backtest_days_count
    / 365
)

baseline_ordering_cost = (
    baseline_number_of_orders
    * ordering_cost_per_order
)

baseline_stockout_cost = (
    baseline_total_lost_sales
    * stockout_cost_per_unit
)

baseline_total_inventory_cost = (
    baseline_holding_cost
    + baseline_ordering_cost
    + baseline_stockout_cost
)

print("Moving Average Cost")
print("-------------------")
print("Holding cost: ₹", round(baseline_holding_cost, 2))
print("Ordering cost: ₹", round(baseline_ordering_cost, 2))
print("Stockout cost: ₹", round(baseline_stockout_cost, 2))
print(
    "Total inventory cost: ₹",
    round(baseline_total_inventory_cost, 2)
)

Moving Average Cost
-------------------
Holding cost: ₹ 27444.38
Ordering cost: ₹ 2000
Stockout cost: ₹ 0
Total inventory cost: ₹ 29444.38


In [81]:
# Final V2 strategy comparison

comparison = pd.DataFrame(
    {
        "Metric": [
            "Average Inventory",
            "Maximum Inventory",
            "Total Units Ordered",
            "Number of Orders",
            "Stockout Days",
            "Lost Sales Units",
            "Service Level (%)",
            "Holding Cost (₹)",
            "Ordering Cost (₹)",
            "Stockout Cost (₹)",
            "Total Inventory Cost (₹)",
        ],
        "XGBoost": [
            average_inventory,
            maximum_inventory,
            total_units_ordered,
            number_of_orders,
            (backtest_results["stockout_units"] > 0).sum(),
            backtest_results["stockout_units"].sum(),
            (
                backtest_results["units_fulfilled"].sum()
                / backtest_results["demand"].sum()
                * 100
            ),
            holding_cost,
            ordering_cost,
            stockout_cost,
            total_inventory_cost,
        ],
        "Moving Average": [
            baseline_avg_inventory,
            baseline_results["closing_stock"].max(),
            baseline_total_units_ordered,
            baseline_number_of_orders,
            baseline_stockout_days,
            baseline_total_lost_sales,
            baseline_service_level,
            baseline_holding_cost,
            baseline_ordering_cost,
            baseline_stockout_cost,
            baseline_total_inventory_cost,
        ],
    }
)

comparison

,Metric,XGBoost,Moving Average
0,Average Inventory,374.923913,544.413043
1,Maximum Inventory,822.000000,1261.000000
2,Total Units Ordered,3949.000000,4139.000000
3,Number of Orders,5.000000,4.000000
4,Stockout Days,2.000000,0.000000
5,Lost Sales Units,26.000000,0.000000
6,Service Level (%),99.329724,100.000000
7,Holding Cost (₹),18900.273973,27444.383562
8,Ordering Cost (₹),2500.000000,2000.000000
9,Stockout Cost (₹),26000.000000,0.000000


In [82]:
comparison 

,Metric,XGBoost,Moving Average
0,Average Inventory,374.923913,544.413043
1,Maximum Inventory,822.000000,1261.000000
2,Total Units Ordered,3949.000000,4139.000000
3,Number of Orders,5.000000,4.000000
4,Stockout Days,2.000000,0.000000
5,Lost Sales Units,26.000000,0.000000
6,Service Level (%),99.329724,100.000000
7,Holding Cost (₹),18900.273973,27444.383562
8,Ordering Cost (₹),2500.000000,2000.000000
9,Stockout Cost (₹),26000.000000,0.000000
